# 05 — Estudo de caso: Rio Claro

**Objetivo:** aprofundar o retrato de Rio Claro dentro do panorama estadual —
o "Nível 2" do estudo (o "Nível 1" é o notebook 04, com os 645 municípios).

**Entrada:** `data/processed/dataset_consolidado_sp.csv` (painel, por ano),
mais os dados específicos de Rio Claro obtidos junto à Prefeitura (seções
5.4 e 5.5 — ver `data/external/FONTES_RIO_CLARO.md`).

**Estrutura desta seção no artigo:** primeiro mostra a associação em escala
estadual (notebook 04), depois usa Rio Claro para dar profundidade — comparar
com a média/mediana do estado, projetar a tendência e detalhar o perfil
socioeconômico dos idosos que moram sozinhos.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv")
rio_claro = df[df["municipio"] == config.RIO_CLARO_NOME].sort_values("ano")
rio_claro


## 5.1 Rio Claro × média e mediana do estado


In [ ]:
resumo_estado = df.groupby("ano").agg(
    media_estado=("taxa_internacao_100k_domicilios_idosos", "mean"),
    mediana_estado=("taxa_internacao_100k_domicilios_idosos", "median"),
).reset_index()

comparativo = resumo_estado.merge(
    rio_claro[["ano", "taxa_internacao_100k_domicilios_idosos"]].rename(
        columns={"taxa_internacao_100k_domicilios_idosos": "rio_claro"}
    ),
    on="ano", how="left",
)
comparativo


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(comparativo["ano"], comparativo["media_estado"], marker="o", label="Média do estado (SP)")
ax.plot(comparativo["ano"], comparativo["rio_claro"], marker="o", label=config.RIO_CLARO_NOME, color="red")
ax.set_xlabel("Ano")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title("Rio Claro × média do estado de SP")
ax.legend()
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "rio_claro_vs_estado.png", dpi=150)
plt.show()


## 5.2 Projeção simples até 2026

Regressão linear simples sobre a série histórica de Rio Claro (2022-2026)
para estimar a tendência — **isso é só uma referência ilustrativa**, não uma
previsão robusta (a série tem poucos pontos, e 2026 está incompleto). Vale
declarar essa limitação no artigo.


In [ ]:
serie = rio_claro.dropna(subset=["taxa_internacao_100k_domicilios_idosos"])
if len(serie) >= 2:
    coefs = np.polyfit(serie["ano"], serie["taxa_internacao_100k_domicilios_idosos"], 1)
    anos_futuros = np.arange(serie["ano"].min(), 2027)
    projecao = np.polyval(coefs, anos_futuros)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(serie["ano"], serie["taxa_internacao_100k_domicilios_idosos"], label="Observado")
    ax.plot(anos_futuros, projecao, "--", color="red", label="Tendência linear (projeção)")
    ax.set_xlabel("Ano")
    ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
    ax.set_title(f"Projeção de tendência — {config.RIO_CLARO_NOME} (ilustrativa)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(config.OUTPUTS_FIGURES / "projecao_rio_claro.png", dpi=150)
    plt.show()
else:
    print("Poucos pontos para projeção — confirme se todos os anos de 2022-2026 foram coletados no notebook 02.")


## 5.3 Perfil das internações em Rio Claro, por causa

Cada "causa" é um capítulo da CID-10 (ver `config.CAUSAS_SIH`), não o
subgrupo específico do plano original — ver notebook 02 para o detalhe.


In [ ]:
causas = list(config.CAUSAS_SIH.keys())
perfil = rio_claro[causas].sum().rename(index=config.CAUSAS_SIH_LABELS)

fig, ax = plt.subplots(figsize=(7, 5))
perfil.sort_values().plot(kind="barh", ax=ax, color="#C44E52")
ax.set_xlabel("Total de internações (2022-2026)")
ax.set_title(f"Perfil das internações em idosos, por capítulo CID-10 — {config.RIO_CLARO_NOME}")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "perfil_causas_rio_claro.png", dpi=150)
plt.show()


## 5.4 Cadastro Único — perfil socioeconômico dos idosos que moram sozinhos

**Fonte:** Ofício SMDS nº 2235/2026, Prefeitura Municipal de Rio Claro,
dados de Junho/2026 (ver `data/external/FONTES_RIO_CLARO.md` para a
citação completa a usar no artigo). Dado agregado por faixa de renda, sem
identificação individual — não há restrição de CEP para este uso.

**Cobertura:** o Cadastro Único registra famílias de baixa renda. Os 8.439
idosos aqui representam ~23% do total de idosos do município (37.038,
Censo 2022) — esta seção descreve o perfil dos idosos **em situação de
vulnerabilidade socioeconômica**, não de todos os idosos de Rio Claro.


In [ ]:
cadunico = pd.read_csv(config.DATA_EXTERNAL / "cadunico_rio_claro.csv")
cadunico["pct_unipessoal"] = (cadunico["idosos_familias_unipessoais"] / cadunico["total_idosos"] * 100).round(1)

total_idosos_cadunico = int(cadunico["total_idosos"].sum())
total_unipessoais_cadunico = int(cadunico["idosos_familias_unipessoais"].sum())
pct_geral_cadunico = total_unipessoais_cadunico / total_idosos_cadunico * 100

print(f"Idosos no CadÚnico de Rio Claro: {total_idosos_cadunico}")
print(f"Em famílias unipessoais (moram sozinhos): {total_unipessoais_cadunico} ({pct_geral_cadunico:.1f}%)")

cadunico


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(cadunico["faixa_renda"], cadunico["pct_unipessoal"], color="#C44E52")
ax.set_ylabel("% de idosos em famílias unipessoais")
ax.set_title(f"{config.RIO_CLARO_NOME} — idosos que moram sozinhos, por faixa de renda\n(CadÚnico, Jun/2026)")
ax.set_ylim(0, 100)
for i, v in enumerate(cadunico["pct_unipessoal"]):
    ax.text(i, v + 2, f"{v}%", ha="center")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "cadunico_pct_unipessoal_por_renda.png", dpi=150)
plt.show()


**Primeira leitura (a aprofundar no texto do artigo):** entre os idosos em
extrema pobreza cadastrados, 75,2% moram sozinhos — a faixa "Baixa Renda"
chama atenção por destoar bastante das vizinhas (10,2%); vale confirmar esse
número com a Prefeitura antes de usar no artigo, pode ser uma
particularidade real da faixa ou uma inconsistência pontual dos dados.

**Comparação com o Censo (notebook 01):** no Censo 2022, 28,5% dos
domicílios de Rio Claro com responsável idoso são unipessoais (todas as
faixas de renda). No CadÚnico (só baixa renda), a proporção geral é bem
maior, 46,1% — consistente com a ideia de que viver sozinho é mais comum
entre idosos de baixa renda, embora as duas fontes meçam populações e
períodos diferentes (Censo 2022, todos os idosos responsáveis por
domicílio; CadÚnico Jun/2026, só idosos de famílias de baixa renda) e não
devam ser somadas ou comparadas como se fossem a mesma métrica — a
comparação vale como leitura qualitativa, não como teste estatístico.


## 5.5 Evolução histórica da população idosa em Rio Claro

**Fonte:** mesmo ofício, Tabela 1 (Censos IBGE 1970-2022) — série oficial
divulgada pela Prefeitura, útil para contextualizar o crescimento do
segmento idoso na introdução do artigo.


In [ ]:
historico = pd.read_csv(config.DATA_EXTERNAL / "censo_rio_claro_historico.csv")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(historico["ano"], historico["pessoas_idosas_pct"], marker="o", color="#4C72B0")
ax.set_xlabel("Ano")
ax.set_ylabel("% de idosos na população")
ax.set_title(f"{config.RIO_CLARO_NOME} — evolução da proporção de idosos (1970-2022)")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "evolucao_idosos_rio_claro.png", dpi=150)
plt.show()

historico


## 5.6 Outros indicadores citados pela Prefeitura (ainda não integrados)

Do mesmo ofício, para uso textual/contextual no artigo (não em tabela ainda):
- **BPC Idoso** (65+, LOAS): 2.130 beneficiários em Rio Claro (Julho/2026)
- **SCFV** (Serviço de Convivência e Fortalecimento de Vínculos): 231
  pessoas idosas atendidas

Se algum desses números ganhar uma seção própria de análise, criamos aqui.
